# Notebook 05 (Lucas): MLP Training -- Exp 2 and Exp 5

**Exp 2:** Delta Residue only -- `mutant_residue_emb[mut_pos] - wt_residue_emb[mut_pos]`  
**Exp 5:** Delta Residue + PCA reduction -- `PCA(delta_residue[mut_pos])`

Both experiments run for both ESM-2 and AbLang2. Training uses MSE loss.
Evaluation metric: Spearman correlation per dataset and aggregate (excluding HER2 separately).
All runs logged to W&B.

## Setup

In [ ]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [ ]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [ ]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

In [ ]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

In [ ]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import wandb
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

## Data Loading and Splits

In [ ]:
df = load_abagym_antibody(DATA_DIR / 'abagym_antibody.csv')

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")
print()

# Verify all datasets are represented in each split
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    counts = df.iloc[idx]['DMS_name'].value_counts().to_dict()
    print(f"{split_name}: {counts}")

Loads the AbAgym metadata CSV and creates the stratified 80/10/10 split.

The split is stratified within each antibody dataset separately, then pooled.
This ensures all 5 antibodies are represented in train, val, and test.
The `random_state=42` is fixed -- Oscar uses the same value in NB05_oscar.ipynb
to guarantee identical splits across both notebooks.

Record confirmed split sizes here after running.

## Experiment 2: Delta Residue Only

**Input:** `mutant_residue_emb[mut_pos] - wt_residue_emb[mut_pos]`  
**Dims:** ESM-2 = 1280, AbLang2 = 480  
**Owner:** Lucas

The most local embedding strategy: a single token-level delta at the mutation site
only. This encodes how much the mutated amino acid shifts the contextual embedding
at that specific position. From NB04 EDA, the L2 norm of this vector is weakly
predictive. The MLP has access to the
full directional vector, not just the norm, so supervised performance should
substantially exceed the EDA baseline.

In [ ]:
# Build datasets for both models -- Exp 2 (DELTA_RESIDUE)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_RESIDUE: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

Sanity check: verifies input dimensions for Exp 2 (ESM-2=1280, AbLang2=480)
and that the dataset loads correctly.

Record confirmed output here after running.

## Experiment 5: Delta Residue + PCA Reduction

**Input:** `PCA(delta_residue[mut_pos])` reduced to `n_components` dimensions  
**Dims:** `n_components` (to be determined -- sweep suggested values below)  
**Owner:** Lucas

Same local delta residue as Exp 2, but dimensionality-reduced via PCA before
the MLP. Motivation: the raw 1280/480-dim vector may have many uninformative
dimensions that add noise. PCA retains the principal axes of variation in the
training set's delta residue space.

**Critical implementation note:** PCA must be fit on the training split only,
then applied to val and test using the same fitted transform. Fitting on the
full dataset would leak test information into the dimensionality reduction.
The `transform` parameter in `AbAgymDataset` handles this: fit PCA on train,
pass `transform=lambda x: pca.transform(x[None])[0]` to all three splits.

Suggested `n_components` to evaluate: 32, 64, 128, 256.

In [ ]:
# Example: fit PCA on training split for ESM-2 delta residue
# Repeat for AbLang2 and for each n_components value

N_COMPONENTS = 64  # adjust as needed
model_name = 'esm2'

# Load raw delta residue for the full dataset to extract train vectors
ds_full = AbAgymDataset(
    antibody_df=df,
    embedding_dir=EMBEDDING_DIR,
    strategy=EmbeddingStrategy.DELTA_RESIDUE,
    model_name=model_name,
)

# Extract training vectors (no transform applied yet)
train_vecs = np.stack([ds_full[i][0].numpy() for i in train_idx])
print(f"Train vectors shape: {train_vecs.shape}")

# Fit PCA on train only
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(train_vecs)
print(f"PCA explained variance (first {N_COMPONENTS} components): {pca.explained_variance_ratio_.sum():.3f}")

# Wrap as transform callable
pca_transform = lambda x: pca.transform(x[None])[0].astype('float32')

# Build dataset with transform applied
ds_reduced = AbAgymDataset(
    antibody_df=df,
    embedding_dir=EMBEDDING_DIR,
    strategy=EmbeddingStrategy.DELTA_RESIDUE_REDUCED,
    model_name=model_name,
    transform=pca_transform,
)
x0, y0, _ = ds_reduced[train_idx[0]]
print(f"Reduced input dim: {x0.shape[0]}")

Fits PCA on the training split only, then wraps it as a transform for the dataset.
The same `pca_transform` is passed to train, val, and test dataset instances to
ensure the same projection is applied consistently.

The explained variance sum tells you how much information is retained at the chosen
`n_components`. A value of 0.8+ is typical for a good reduction.

Record confirmed output (train shape, explained variance, reduced dim) here after running.